# Train BiPO Steering Vector

In [2]:
from pathlib import Path
import sys

import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from steering.bipo import load_model, steering

In [3]:
if torch.cuda.is_available():
    device, dtype = "cuda", torch.bfloat16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.bfloat16

In [4]:
model, tokenizer = load_model()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1107 05:24:50.570000 4250 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
TRAITS = ('openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism')

In [ ]:
for trait in TRAITS:
    steering_vector = steering(
        model,
        trait,
        tokenizer=tokenizer,
        data_dir=REPO_ROOT / "data" / "pairs",
        epochs=1,
        chat_template=True,
    )
    output_dir = REPO_ROOT / "artifacts" / "steering_vectors" / "bipo"
    output_path = output_dir / f"{trait}-2.pt"
    torch.save(steering_vector.cpu(), output_path)
    print(f"Saved steering vector to {output_path}")
    print(f"Shape: {tuple(steering_vector.shape)}")

openness: rows=330 train=0 dev=10 test=0 missing=320
openness: final train=256 dev=74
Saved steering vector to /Users/arya/Documents/GT/CS7650 - NLP/Project/llm-personality-conditioning/artifacts/steering_vectors/bipo/openness-2.pt
Shape: (4096,)
conscientiousness: rows=281 train=181 dev=100 test=0 missing=0
conscientiousness: final train=181 dev=100
Saved steering vector to /Users/arya/Documents/GT/CS7650 - NLP/Project/llm-personality-conditioning/artifacts/steering_vectors/bipo/conscientiousness-2.pt
Shape: (4096,)
extraversion: rows=288 train=188 dev=100 test=0 missing=0
extraversion: final train=188 dev=100


In [ ]:
import gc

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

NameError: name 'model' is not defined